## 1. OVERVIEW & PREPROCESSING

In [102]:
import pandas as pd
import numpy as np
import os
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold


### 1.1. Dataset Overview

In [78]:
train_df = pd.read_csv("dataset/raw/train.csv")
test_df  = pd.read_csv("dataset/raw/test.csv")

In [4]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2492196 entries, 0 to 2492195
Data columns (total 28 columns):
 #   Column             Dtype  
---  ------             -----  
 0   ID                 object 
 1   Severity           int64  
 2   Start_Time         object 
 3   End_Time           object 
 4   Latitude           float64
 5   Longitude          float64
 6   Distance(mi)       float64
 7   Description        object 
 8   Street             object 
 9   City               object 
 10  County             object 
 11  State              object 
 12  Zipcode            object 
 13  Timezone           object 
 14  Airport_Code       object 
 15  Weather_Timestamp  object 
 16  Temperature(F)     float64
 17  Humidity(%)        float64
 18  Visibility(mi)     float64
 19  Weather_Condition  object 
 20  Amenity            bool   
 21  Crossing           bool   
 22  Junction           bool   
 23  Railway            bool   
 24  Station            bool   
 25  Stop              

In [5]:
train_df.head()

,ID,Severity,Start_Time,End_Time,Latitude,Longitude,Distance(mi),Description,Street,City,...,Visibility(mi),Weather_Condition,Amenity,Crossing,Junction,Railway,Station,Stop,Traffic_Signal,Sunrise_Sunset
0,A-4081224,4,2022-03-18 06:55:00,2022-03-18 11:55:00,36.081864,-79.080223,0.039,The road is closed near St Mary's Rd.,St Marys Rd,Hillsborough,...,NaN,NaN,False,False,False,False,False,False,False,Night
1,A-4338502,2,2023-01-29 16:35:00,2023-01-29 17:53:04,38.927944,-121.056469,0.014,Incident on BOWMAN RD near LUTHER RD Drive wit...,Bowman Rd,Auburn,...,10.0,Cloudy,False,False,False,False,False,True,False,Day
2,A-4813526,2,2022-10-11 12:53:40,2022-10-11 14:44:43,37.539693,-77.432146,0.008,Incident on N 11TH ST near E BROAD ST RICH Dri...,E Broad St,Richmond,...,10.0,Fair,False,True,False,False,False,False,True,Day
3,A-4731406,2,2022-01-24 06:12:00,2022-01-24 06:40:30,36.998262,-76.409113,0.944,Slow traffic on I-664 S from Aberdeen Rd/Exit ...,I-664 S,Newport News,...,10.0,Fair,False,False,True,False,False,False,False,Night
4,A-7263275,2,2020-02-14 15:28:00,2020-02-14 16:26:09,38.524770,-121.467450,0.000,At Fruitridge Rd - Accident.,CA-99 N,Sacramento,...,10.0,Fair,False,False,True,False,False,False,False,Day


In [6]:
train_df.isnull().sum()

ID                       0
Severity                 0
Start_Time               0
End_Time                 0
Latitude                 0
Longitude                0
Distance(mi)             0
Description              0
Street                5411
City                   100
County                   0
State                    0
Zipcode                724
Timezone              2782
Airport_Code          9295
Weather_Timestamp    47950
Temperature(F)       62100
Humidity(%)          65945
Visibility(mi)       64006
Weather_Condition    62172
Amenity                  0
Crossing                 0
Junction                 0
Railway                  0
Station                  0
Stop                     0
Traffic_Signal           0
Sunrise_Sunset       12720
dtype: int64

In [79]:
def show_unique_values(df, columns=None, max_display=50):
    """
    Display the unique values of columns in a DataFrame
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The DataFrame to analyze
    columns : list, optional
        A list of columns to display. If None, all columns will be displayed
    max_display : int, default=50
        The maximum number of unique values to display for each column
        
    Returns:
    --------
    dict
        A dictionary containing unique values for each column
    """
    # If columns are not specified, take all columns
    if columns is None:
        columns = df.columns.tolist()
    
    # Dictionary to store results
    unique_dict = {}
    
    print("=" * 100)
    print("UNIQUE VALUES ANALYSIS")
    print("=" * 100)
    
    for col in columns:
        if col not in df.columns:
            print(f"Column '{col}' does not exist in the DataFrame")
            continue
        
        unique_vals = df[col].unique()
        n_unique = len(unique_vals)
        
        # Get value counts
        value_counts = df[col].value_counts(dropna=False)
        
        # Store in the dictionary
        unique_dict[col] = unique_vals
        
        # Print information
        print(f"\nFeature: {col}")
        print(f"   • Data type: {df[col].dtype}")
        print(f"   • Total unique values: {n_unique}")
        print(f"   • Missing values: {df[col].isnull().sum()} ({df[col].isnull().sum()/len(df)*100:.2f}%)")
        
        # Display unique values
        if n_unique <= max_display:
            print(f"   • Unique values: {sorted([str(v) for v in unique_vals if pd.notna(v)])}")
            print(f"\n   Value Counts:")
            for val, count in value_counts.head(20).items():
                pct = (count / len(df)) * 100
                print(f"      {str(val):<30} : {count:>6} ({pct:>5.2f}%)")
        else:
            print(f"   • Too many unique values ({n_unique}). Showing top 20:")
            for val, count in value_counts.head(20).items():
                pct = (count / len(df)) * 100
                print(f"      {str(val):<30} : {count:>6} ({pct:>5.2f}%)")
        
        print("-" * 100)
    
    return unique_dict


In [80]:
# Ví dụ 3: Xem unique values của tất cả các cột
unique_all = show_unique_values(train_df)

UNIQUE VALUES ANALYSIS

Feature: ID
   • Data type: object
   • Total unique values: 2492196
   • Missing values: 0 (0.00%)
   • Too many unique values (2492196). Showing top 20:
      A-4081224                      :      1 ( 0.00%)
      A-4338502                      :      1 ( 0.00%)
      A-4813526                      :      1 ( 0.00%)
      A-4731406                      :      1 ( 0.00%)
      A-7263275                      :      1 ( 0.00%)
      A-7362456                      :      1 ( 0.00%)
      A-4967672                      :      1 ( 0.00%)
      A-5506850                      :      1 ( 0.00%)
      A-7262705                      :      1 ( 0.00%)
      A-6988745                      :      1 ( 0.00%)
      A-7195257                      :      1 ( 0.00%)
      A-5409238                      :      1 ( 0.00%)
      A-7139936                      :      1 ( 0.00%)
      A-4428004                      :      1 ( 0.00%)
      A-3763270                      :      1 ( 0.0

### 1.2. Convert to the correct type

In [81]:
# Convert time columns to datetime

time_col = ["Start_Time", "End_Time", "Weather_Timestamp"]

for col in time_col:
    train_df[col] = pd.to_datetime(train_df[col], errors='coerce')
    test_df[col] = pd.to_datetime(test_df[col], errors='coerce')

In [82]:
# Convert bool columns to object

bool_col = ['Amenity', 'Crossing', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal']

for col in bool_col:
    train_df[col] = train_df[col].astype('object')
    test_df[col] = test_df[col].astype('object')

In [83]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2492196 entries, 0 to 2492195
Data columns (total 28 columns):
 #   Column             Dtype         
---  ------             -----         
 0   ID                 object        
 1   Severity           int64         
 2   Start_Time         datetime64[ns]
 3   End_Time           datetime64[ns]
 4   Latitude           float64       
 5   Longitude          float64       
 6   Distance(mi)       float64       
 7   Description        object        
 8   Street             object        
 9   City               object        
 10  County             object        
 11  State              object        
 12  Zipcode            object        
 13  Timezone           object        
 14  Airport_Code       object        
 15  Weather_Timestamp  datetime64[ns]
 16  Temperature(F)     float64       
 17  Humidity(%)        float64       
 18  Visibility(mi)     float64       
 19  Weather_Condition  object        
 20  Amenity            objec

### 1.3. Define X and y

In [84]:
X_train = train_df.drop("Severity", axis=1)
y_train = train_df["Severity"]

X_test = test_df.drop("Severity", axis=1)
y_test = test_df["Severity"]

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (2492196, 27)
y_train shape: (2492196,)
X_test shape: (623050, 27)
y_test shape: (623050,)


### 1.4. Remove Zero Variance Features and Duplicated Rows

In [86]:
class VarianceThresholdSelector(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0):
        self.threshold = threshold
        self.selector = VarianceThreshold(threshold=self.threshold)
    
    def fit(self, X, y=None):
        numeric_cols = X.select_dtypes(include=['float64', 'int64']).columns
        self.numeric_cols = numeric_cols
        self.selector.fit(X[numeric_cols])
        # Save retained numeric columns
        self.retained_cols = numeric_cols[self.selector.get_support()]
        # Save dropped numeric columns
        self.dropped_cols = [col for col in numeric_cols if col not in self.retained_cols]
        print("Columns removed due to zero variance:", self.dropped_cols)
        return self
    
    def transform(self, X):
        cols_to_keep = list(self.retained_cols) + list(X.columns.difference(self.numeric_cols))
        return X[cols_to_keep]

In [87]:
class ConstantAndDuplicateRemover(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Identify constant columns
        self.constant_cols = [col for col in X.columns if X[col].nunique() == 1]
        print("Columns removed due to having only 1 unique value:", self.constant_cols)
        return self
    
    def transform(self, X):
        # Drop constant columns
        X_cleaned = X.drop(columns=self.constant_cols, errors="ignore")
        # Drop duplicate rows
        X_cleaned = X_cleaned.drop_duplicates()
        return X_cleaned

In [88]:
cleaning_pipeline = Pipeline(steps=[
    ("var_thresh", VarianceThresholdSelector(threshold=0)),
    ("const_dup", ConstantAndDuplicateRemover())

])
X_train_cleaned = cleaning_pipeline.fit_transform(X_train.copy())
print(f"Data shape after cleaning: {X_train_cleaned.shape}")
X_test_cleaned = cleaning_pipeline.transform(X_test.copy())
print(f"Data shape after cleaning: {X_test_cleaned.shape}")

Columns removed due to zero variance: []
Columns removed due to having only 1 unique value: []
Data shape after cleaning: (2492196, 27)
Data shape after cleaning: (623050, 27)


c:\Users\dangn\.conda\envs\dpav_env\Lib\site-packages\sklearn\pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


### 1.5. Feature Dropping Strategy

We systematically remove features that cause data leakage, have excessive missing values, or provide redundant/irrelevant information for predicting accident severity.

#### a. Data Leakage (Target Information)
Features that directly reveal the severity outcome must be removed:
- `ID`: Unique identifier with no predictive value
- `Description`: Text containing severity keywords (e.g., "serious accident", "minor incident")
- `End_Time`: Duration (End_Time - Start_Time) directly correlates with severity - more severe accidents take longer to clear
- `Distance(mi)`: Affected road length is a direct consequence of severity, not a predictor

#### b. Redundant Location Features
High-cardinality location features that are too specific or redundant:
- `Street`: Extremely high cardinality (10,000+ unique values); already captured by City/County/State
- `Zipcode`: Overly granular; aggregate location (City/County) is sufficient
- `Latitude` / `Longitude`: Exact coordinates add noise without explanatory power for severity; aggregate location features are more generalizable

#### c. Metadata & Auxiliary Information
Features that provide no predictive signal:
- `Airport_Code`: Weather station identifier; actual weather measurements are already included
- `Timezone`: Redundant with State
- `Weather_Timestamp`: Exact timestamp of weather reading is unnecessary; weather values themselves are what matter

**Total features to drop: 11**

In [89]:
print("=" * 90)
print("FEATURE DROPPING STRATEGY")
print("=" * 90)

drop_leakage = ['ID', 'Description', 'End_Time', 'Distance(mi)']
drop_location = ['Street', 'Zipcode', 'Latitude', 'Longitude']
drop_metadata = ['Airport_Code', 'Timezone', 'Weather_Timestamp']

all_cols_to_drop = drop_leakage + drop_location + drop_metadata

# Filter only existing columns
cols_to_drop = [col for col in all_cols_to_drop if col in X_train.columns]

print(f"\nDropping {len(cols_to_drop)} features from {X_train.shape[1]} total features:\n")

# Apply drops to both train and test sets
X_train = X_train.drop(columns=cols_to_drop, errors='ignore')
X_test = X_test.drop(columns=cols_to_drop, errors='ignore')

print(f"   • Features dropped: {len(cols_to_drop)}")
print(f"   • Train shape: {X_train.shape}")
print(f"   • Test shape:  {X_test.shape}")

FEATURE DROPPING STRATEGY

Dropping 11 features from 27 total features:

   • Features dropped: 11
   • Train shape: (2492196, 16)
   • Test shape:  (623050, 16)


### 1.6. Define Categorical and Numerical Variables


In [90]:
numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

In [91]:
print(f" Number of numerical features: {len(numeric_features)}")
print(f" List: {numeric_features}\n")

print(f" Number of categorical features: {len(categorical_features)}")
print(f" List: {categorical_features}\n")


 Number of numerical features: 3
 List: ['Temperature(F)', 'Humidity(%)', 'Visibility(mi)']

 Number of categorical features: 12
 List: ['City', 'County', 'State', 'Weather_Condition', 'Amenity', 'Crossing', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal', 'Sunrise_Sunset']



# 2. MISSING VALUES PROCESSING

### 2.1. Detaild Missing Values Analysis

In [92]:
# Detailed analysis by feature groups
print("=" * 90)
print("DETAILED MISSING VALUES ANALYSIS")
print("=" * 90)

# Determine feature type
def get_feature_type(col):
    if col in numeric_features:
        return 'Numerical'
    elif col in categorical_features:
        return 'Categorical'
    else:
        return 'Other'

# Calculate statistics
missing_info = pd.DataFrame({
    'Feature': X_train_cleaned.columns,
    'Feature Type': [get_feature_type(col) for col in X_train_cleaned.columns],
    'Missing Count': X_train_cleaned.isnull().sum().values,
    'Missing %': (X_train_cleaned.isnull().sum() / len(X_train_cleaned) * 100).values,
    'Present Count': X_train_cleaned.notnull().sum().values,
    'Data Type': X_train_cleaned.dtypes.values
})

# Classify by missing level
missing_info['Missing Level'] = pd.cut(
    missing_info['Missing %'],
    bins=[-0.1, 0, 5, 15, 100],
    labels=['None', 'Low (0-5%)', 'Medium (5-15%)', 'High (>15%)']
)

# Display only features with missing values
missing_only = missing_info[missing_info['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print("\nFEATURES WITH MISSING VALUES:\n")
print(f"{'Feature':<30} | {'Type':<15} | {'Missing':<12} | {'%':<8} | {'Level':<15}")
print("-" * 95)
for idx, row in missing_only.iterrows():
    print(f"{row['Feature']:<30} | {row['Feature Type']:<15} | {row['Missing Count']:>10,} | {row['Missing %']:>6.2f}% | {row['Missing Level']:<15}")

# Overall statistics
print("\n" + "=" * 90)
print("OVERALL STATISTICS:")
print(f"   • Total observations: {len(X_train_cleaned):,}")
print(f"   • Total features: {len(X_train_cleaned.columns)}")
print(f"   • Features with missing: {len(missing_only)}")
print(f"   • Features without missing: {len(X_train_cleaned.columns) - len(missing_only)}")
print(f"   • Total missing cells: {X_train_cleaned.isnull().sum().sum():,}")
print(f"   • Overall missing rate: {(X_train_cleaned.isnull().sum().sum() / (len(X_train_cleaned) * len(X_train_cleaned.columns)) * 100):.2f}%")
print("=" * 90)

DETAILED MISSING VALUES ANALYSIS

FEATURES WITH MISSING VALUES:

Feature                        | Type            | Missing      | %        | Level          
-----------------------------------------------------------------------------------------------
Humidity(%)                    | Numerical       |     65,945 |   2.65% | Low (0-5%)     
Visibility(mi)                 | Numerical       |     64,006 |   2.57% | Low (0-5%)     
Weather_Condition              | Categorical     |     62,172 |   2.49% | Low (0-5%)     
Temperature(F)                 | Numerical       |     62,100 |   2.49% | Low (0-5%)     
Weather_Timestamp              | Other           |     47,950 |   1.92% | Low (0-5%)     
Sunrise_Sunset                 | Categorical     |     12,720 |   0.51% | Low (0-5%)     
Airport_Code                   | Other           |      9,295 |   0.37% | Low (0-5%)     
Street                         | Other           |      5,411 |   0.22% | Low (0-5%)     
Timezone                  

`City` is a key location feature with very low missing rate (<1%), so we drop rows with missing City values to preserve data quality instead of imputing.

In [93]:
# Drop rows with missing City values
train_missing_city = X_train_cleaned[X_train_cleaned['City'].isnull()].index

X_train_cleaned = X_train_cleaned.drop(train_missing_city)
y_train = y_train.drop(train_missing_city)

print(f"Dropped {len(train_missing_city):,} rows with missing City from training set")
print(f"New X_train_cleaned shape: {X_train_cleaned.shape}")
print(f"New y_train shape: {y_train.shape}")

Dropped 100 rows with missing City from training set
New X_train_cleaned shape: (2492096, 27)
New y_train shape: (2492096,)


## 3. Outlier Processing

### 3.1. Detaild Outlier Values Analysis

## Outlier Analysis (IQR Method)
### Objective
Detect and handle abnormal values in the dataset to ensure data quality for analysis and modeling.
### Method
**IQR Calculation:**  
IQR is calculated for each numeric column:
$$
IQR = Q3 - Q1
$$
**Outlier Bounds:**  
The lower and upper bounds are:
$$
\text{Lower Bound} = Q1 - 1.5 \times IQR
$$
$$
\text{Upper Bound} = Q3 + 1.5 \times IQR
$$
Values outside this range are considered **outliers**.
### Reason for Using IQR
IQR identifies outliers based on statistical distribution, minimizing the influence of extreme values, and provides an objective method for numeric variables.
### Benefits
- Removes abnormal values, ensuring clean and reliable data.  
- Improves accuracy and stability of analysis and machine learning models.  
- Prepares data for downstream preprocessing and modeling.


In [94]:
outlier_cols = ['Temperature(F)', 'Humidity(%)', 'Visibility(mi)']

print("\n" + "=" * 90)
print("OUTLIER ANALYSIS (IQR METHOD)")
print("=" * 90)

# Check cột tồn tại
available_cols = [c for c in outlier_cols if c in X_train_cleaned.columns]
report_data = []

for col in available_cols:
    # 1. Calculate IQR
    
    Q1 = X_train_cleaned[col].quantile(0.25)
    Q3 = X_train_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1
    
    # 2. Define Limits (Tính ngưỡng thống kê ban đầu)
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
        
    # 3. Count outliers (Đếm dựa trên ngưỡng mới)
    outliers_mask = (X_train_cleaned[col] < lower_bound) | (X_train_cleaned[col] > upper_bound)
    num_outliers = outliers_mask.sum()
    pct_outliers = (num_outliers / len(X_train_cleaned)) * 100
    
    if num_outliers > 0:
        report_data.append({
            'Feature': col,
            'Count': num_outliers,
            'Percent': f"{pct_outliers:.2f}%",
            'Rec. Lower': f"{lower_bound:.2f}",
            'Rec. Upper': f"{upper_bound:.2f}",
            'Min': f"{X_train_cleaned[col].min():.2f}",
            'Max': f"{X_train_cleaned[col].max():.2f}"
        })

# 4. Display Report Table
if len(report_data) > 0:
    report_df = pd.DataFrame(report_data)
    print(f"Outliers detected in {len(report_data)} columns:\n")
    # Định dạng bảng đẹp
    print(report_df.to_string(index=False, justify='left'))
else:
    print("No significant outliers found using the IQR method.")

# --- OVERALL STATISTICS SECTION ---
if report_data:
    total_outlier_cells = sum(item['Count'] for item in report_data)
    total_features_analyzed = len(available_cols)
    total_cells_analyzed = len(X_train_cleaned) * total_features_analyzed

    print("\n" + "=" * 90)
    print("OVERALL OUTLIER STATISTICS:")
    print(f"   • Total observations: {len(X_train_cleaned):,}")
    print(f"   • Features analyzed: {total_features_analyzed}")
    print(f"   • Features with outliers: {len(report_data)}")
    print(f"   • Total outlier cells: {total_outlier_cells:,}")
    if total_cells_analyzed > 0:
        print(f"   • Overall outlier rate: {(total_outlier_cells / total_cells_analyzed * 100):.2f}%")
    print("=" * 90)


OUTLIER ANALYSIS (IQR METHOD)
Outliers detected in 2 columns:

Feature         Count Percent Rec. Lower Rec. Upper Min    Max   
Temperature(F)  15524  0.62%   6.00      118.00     -89.00 207.00
Visibility(mi) 462300 18.55%  10.00       10.00       0.00 140.00

OVERALL OUTLIER STATISTICS:
   • Total observations: 2,492,096
   • Features analyzed: 3
   • Features with outliers: 2
   • Total outlier cells: 477,824
   • Overall outlier rate: 6.39%


## Outlier Analysis Results

The outlier analysis highlights anomalies in two key meteorological features: **Temperature(F)** and **Visibility(mi)**.

* **Temperature(F):** Outlier rate is low (0.63%). Extreme values like **-89°F** and **207°F** are physically impossible for traffic conditions, indicating sensor or data entry errors.
* **Visibility(mi):** Outlier rate is high (**18.54%**). Most flagged values (below 10 miles) reflect real adverse weather (fog, heavy rain) rather than errors.

### 3.2. Outlier Processing

In [21]:
# class IQROutlierCapper(BaseEstimator, TransformerMixin):
#     def __init__(self, factor=1.5):
#         self.factor = factor
#         self.lower_bound_ = {}
#         self.upper_bound_ = {}
        
#     def fit(self, X, y=None):
#         if isinstance(X, np.ndarray):
#             X = pd.DataFrame(X)
            
#         for col in X.columns:
#             Q1 = X[col].quantile(0.25)
#             Q3 = X[col].quantile(0.75)
#             IQR = Q3 - Q1
            
#             lower = Q1 - self.factor * IQR
#             upper = Q3 + self.factor * IQR
            
#             # Giới hạn riêng cho Vict Age
#             if col == 'Vict Age':
#                 lower = max(lower, 0)
#                 upper = min(upper, 100)  # Không quá 100 tuổi
                
#             self.lower_bound_[col] = lower
#             self.upper_bound_[col] = upper
#         return self
    
#     def transform(self, X):
#         X_copy = X.copy()
#         if isinstance(X_copy, np.ndarray):
#             X_copy = pd.DataFrame(X_copy)
            
#         for col in X_copy.columns:
#             if col in self.upper_bound_:
#                 X_copy[col] = np.clip(X_copy[col], self.lower_bound_[col], self.upper_bound_[col])
#         return X_copy
#Visibility: GIỮ NGUYÊN - outliers là real weather conditions
# Low visibility (<10 mi) là yếu tố quan trọng gây tai nạn

## Outlier Handling Strategy
Instead of using statistical IQR bounds for all features, we apply **physical constraints** based on real-world limits:

**Temperature(F):** Capped to **[-50°F, 130°F]**  
- Statistical outliers like -89°F or 207°F are physically impossible for traffic conditions
- These values indicate sensor errors or data entry mistakes
- Physical limits preserve realistic weather conditions while removing invalid data

**Humidity(%):** Capped to **[0%, 100%]**  
- Humidity is a percentage measurement with strict physical bounds
- Values outside this range are measurement errors

**Visibility(mi):** No capping applied  
- High outlier rate (18.54%) represents **real adverse weather conditions** (fog, heavy rain, snow)
- Low visibility values are legitimate accident risk factors, not errors
- Capping would remove valuable information about dangerous driving conditions

This approach balances **data quality** (removing impossible values) with **domain knowledge** (preserving meaningful extreme conditions).

In [ ]:
class DomainBasedOutlierCapper(BaseEstimator, TransformerMixin):
    """
    Domain-based outlier handler cho accident data.
    Sử dụng physical limits thay vì IQR cho một số features.
    """
    def __init__(self, factor=1.5):
        self.factor = factor
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        if isinstance(X_copy, np.ndarray):
            X_copy = pd.DataFrame(X_copy)
        
        # Temperature: Clip physical limits (-50°F to 130°F)
        if 'Temperature(F)' in X_copy.columns:
            X_copy['Temperature(F)'] = X_copy['Temperature(F)'].clip(-50, 130)
            
        # Humidity: Physical limits (0% to 100%)
        if 'Humidity(%)' in X_copy.columns:
            X_copy['Humidity(%)'] = X_copy['Humidity(%)'].clip(0, 100)
        return X_copy

## DomainBasedOutlierCapper Class

`DomainBasedOutlierCapper` is a custom scikit-learn transformer for **capping outliers using physical constraints** instead of statistical methods.

### How It Works
- **Temperature(F)**: Clips values to physically realistic range [-50°F, 130°F]
- **Humidity(%)**: Clips values to valid percentage range [0%, 100%]
- **Visibility(mi)**: No capping applied (low visibility is real weather condition, not an error)

### Benefits
- Removes physically impossible values (sensor errors, data entry mistakes)
- Preserves meaningful extreme conditions (adverse weather)
- Domain knowledge-based approach ensures data quality without losing valuable information
- Easily integrates into scikit-learn preprocessing pipelines

In [ ]:
# filter existing columns
outlier_cols = [c for c in outlier_cols if c in X_train_cleaned.columns]
print("\nOutlier Processing Columns:", outlier_cols)


Outlier Processing Columns: ['Temperature(F)', 'Humidity(%)', 'Visibility(mi)']


In [24]:
# # Pipeline xử lý outlier
# outlier_pipeline = Pipeline(steps=[
#     ('capper', IQROutlierCapper(factor=1.5))
# ])

# preprocessor = ColumnTransformer(
#     transformers=[
#         ('outlier_handling', outlier_pipeline, outlier_cols)
#     ],
#     remainder='passthrough',
#     verbose_feature_names_out=False
# )

# print("\n--- Đang xử lý dữ liệu (Outlier Capping)... ---")

# X_train_cleaned_capped = preprocessor.fit_transform(X_train_cleaned)
# X_test_cleaned_capped = preprocessor.transform(X_test_cleaned)


# print("Xử lý hoàn tất!")
# print("-" * 90)

## 4. Full Pipeline
This code implements a **sequential preprocessing pipeline** that handles both missing values and outliers using domain-based constraints.

### Pipeline Structure

**Step 1: Missing Value Imputation**
- Uses `SimpleImputer` with median strategy
- Applied to all numeric features
- Preserves data integrity without introducing bias

**Step 2: Domain-Based Outlier Capping**
- Uses `DomainBasedOutlierCapper` with physical limits
- Applied only to specific weather features (Temperature, Humidity)
- Based on real-world constraints, not statistical bounds

### Implementation Details
- `SequentialNumericProcessor` custom transformer ensures proper execution order
- Maintains all numeric columns in DataFrame format throughout processing
- Applies transformations consistently to both train and test sets
- Returns cleaned datasets (`X_train_cleaned_capped`, `X_test_cleaned_capped`) with original column names

### Benefits
- **Two-stage approach**: Handles missing values first, then caps outliers
- **No data loss**: All rows preserved, only extreme values adjusted
- **Domain knowledge integration**: Physical constraints ensure data validity
- **Scikit-learn compatible**: Easy integration into ML workflows

In [98]:
# Pipeline mising value imputation
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

In [99]:
# Pipeline outlier capping with physical limits
outlier_pipeline = Pipeline(steps=[
    ('capper', DomainBasedOutlierCapper(factor=1.5))
])

In [100]:
class SequentialNumericProcessor(BaseEstimator, TransformerMixin):
    def __init__(self, numeric_features, outlier_cols):
        self.numeric_features = numeric_features
        self.outlier_cols = outlier_cols
        self.imputer = SimpleImputer(strategy='median')
        self.capper = DomainBasedOutlierCapper(factor=1.5)
        
    def fit(self, X, y=None):
        if len(self.numeric_features) > 0:
            self.imputer.fit(X[self.numeric_features])
        self.capper.fit(X[self.outlier_cols] if len(self.outlier_cols) > 0 else X[[]])
        return self
    
    def transform(self, X):
        X_result = X.copy()
        
        # Step 1: Impute missing values for numeric features
        if len(self.numeric_features) > 0:
            X_result[self.numeric_features] = self.imputer.transform(X_result[self.numeric_features])
        
        # Step 2: Cap outliers for specific columns
        if len(self.outlier_cols) > 0:
            X_result[self.outlier_cols] = self.capper.transform(X_result[self.outlier_cols])
        
        return X_result

# Create preprocessor
preprocessor = SequentialNumericProcessor(
    numeric_features=numeric_features,
    outlier_cols=outlier_cols
)

In [101]:
print("\n--- Processing (Missing Values & Outlier Capping)... ---")

# Fit and transform
X_train_cleaned_capped = preprocessor.fit_transform(X_train_cleaned)
X_test_cleaned_capped = preprocessor.transform(X_test_cleaned)

# Data is already a DataFrame, no need to convert
print("Processing complete!")
print(f"✓ Train shape: {X_train_cleaned_capped.shape}")
print(f"✓ Test shape: {X_test_cleaned_capped.shape}")
print("-" * 90)


--- Processing (Missing Values & Outlier Capping)... ---
Processing complete!
✓ Train shape: (2492096, 27)
✓ Test shape: (623050, 27)
------------------------------------------------------------------------------------------


In [103]:
# Create processed directory
os.makedirs("dataset/processed", exist_ok=True)

train_final = X_train_cleaned_capped.copy()
train_final['Severity'] = y_train

test_final = X_test_cleaned_capped.copy()
test_final['Severity'] = y_test

# Save CSV files
train_final.to_csv("dataset/processed/train_processed.csv", index=False)
test_final.to_csv("dataset/processed/test_processed.csv", index=False)

print(f"\n✓ Train: dataset/processed/train_processed.csv")
print(f"  → {train_final.shape[0]:,} rows × {train_final.shape[1]} columns")
print(f"  → Features: {train_final.shape[1] - 1}, Target: Severity")

print(f"\n✓ Test:  dataset/processed/test_processed.csv")
print(f"  → {test_final.shape[0]:,} rows × {test_final.shape[1]} columns")
print(f"  → Features: {test_final.shape[1] - 1}, Target: Severity")

print("\n📊 Data already for next step!")
print("="*90)


✓ Train: dataset/processed/train_processed.csv
  → 2,492,096 rows × 28 columns
  → Features: 27, Target: Severity

✓ Test:  dataset/processed/test_processed.csv
  → 623,050 rows × 28 columns
  → Features: 27, Target: Severity

📊 Data already for next step!
